#Load Zip Dataset

In [ ]:
from google.colab import files

uploaded = files.upload()

# Extracting Zip File

In [ ]:
import zipfile

zip_ref = zipfile.ZipFile('/content/Vehicles-Dataset.zip', 'r')
zip_ref.extractall('/content')
zip_ref.close()

In [ ]:
!pip install split-folders

In [ ]:
import splitfolders

splitfolders.ratio("/content/Vehicles-Dataset", output="/content/vehicles_dataset_split", seed=42, ratio=(.8, .2))

#Preparing Dataset

In [ ]:
from tensorflow import keras
import tensorflow as tf

# generators
train_ds = keras.utils.image_dataset_from_directory(
    directory = '/content/vehicles_dataset_split/train',
    labels = 'inferred',
    label_mode = 'int',
    batch_size = 32,
    image_size = (128,128),
    color_mode='rgb',
    shuffle=True
)


val_ds = keras.utils.image_dataset_from_directory(
    directory = '/content/vehicles_dataset_split/val',
    labels = 'inferred',
    label_mode = 'int',
    batch_size = 32,
    image_size = (128,128),
    color_mode='rgb',
    shuffle=True
)

#Image Visualization

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Get class names
class_names = train_ds.class_names

# Extract one batch from the dataset
for images, labels in train_ds.take(1):
    images = images.numpy().astype("uint8")
    labels = labels.numpy()

    # Set up subplot grid
    plt.figure(figsize=(10, 10))

    for i in range(9):  # First 9 images
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i])
        plt.title(class_names[labels[i]])
        plt.axis("off")

    plt.tight_layout()
    plt.show()

#Model Building

In [ ]:
from keras import Sequential
from keras.layers import Dense,Conv2D,MaxPooling2D,Flatten,BatchNormalization,Input, Dropout

model = Sequential()
model.add(Input(shape=(128,128,3)))

model.add(Conv2D(32, kernel_size=(3, 3),activation='relu',padding="same",))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(3, 3),padding='same'))


model.add(Conv2D(64, kernel_size=(3, 3),activation='relu',padding="same"))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(3, 3),padding='same'))


model.add(Conv2D(128, kernel_size=(3, 3),activation='relu',padding="same"))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(3, 3),padding='same'))


model.add(Flatten())
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(4, activation='softmax'))

model.summary()

model.compile(optimizer='Adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

#Model Training

In [ ]:
epochs=20
history = model.fit(train_ds, epochs=epochs, validation_data=val_ds)

#Plot Accuracy and Loss

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'val'], loc='upper left')
plt.show()

plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'val'], loc='upper left')
plt.show()

#Validation Accuracy

In [ ]:
val_loss, val_accuracy = model.evaluate(val_ds)

print(f"Validation Accuracy: {val_accuracy:.4f}")

#Confusion Matrix & Classification Report

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# Get class names
class_names = val_ds.class_names

# Step 1: Collect true and predicted labels
y_true = []
y_pred = []

for images, labels in val_ds:
    preds = model.predict(images)
    preds = np.argmax(preds, axis=1)  # get class with highest probability

    y_true.extend(labels.numpy())
    y_pred.extend(preds)

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Step 2: Print classification report
print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

# Step 3: Confusion matrix
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()

# Gradio Web App"""

In [ ]:
pip install gradio

import gradio as gr
import tensorflow as tf
import numpy as np



# Prediction function
def predict(image):
    # Resize and normalize image
    img = tf.image.resize(image, (128, 128))  # match model input size
    img = tf.expand_dims(img, axis=0)         # add batch dimension


    # Predict
    pred = model.predict(img)
    class_idx = np.argmax(pred[0])
    class_name = class_names[class_idx]
    confidence = float(pred[0][class_idx])

    return {class_name: confidence}

# Gradio interface
interface = gr.Interface(
    fn=predict,
    inputs=gr.Image(type="numpy"),
    outputs=gr.Label(num_top_classes=3),
    title="Vehicle Classification",
    description="Upload a vehicle image to classify it."
)

# Launch the app
interface.launch()